# 📈 Business Analytics — Ejercicios con Soluciones

Este notebook cubre dos pilares del análisis de negocios:
- **Ejercicio 1:** Series de tiempo con Statsmodels (tendencias, estacionalidad, descomposición y forecast)
- **Ejercicio 2:** Dashboard interactivo de KPIs empresariales con Plotly

> **Nota sobre Tableau:** El ejercicio original propone Tableau, que requiere licencia y no corre en notebooks.
> Lo reemplazamos con **Plotly**, que ofrece gráficos interactivos equivalentes, es 100% Python y gratuito.
> Los conceptos de KPIs, filtros y dashboards son exactamente los mismos.

**Librerías necesarias:**
```bash
pip install statsmodels plotly pandas numpy matplotlib
```

**Cómo usar este notebook:**
1. Lee el **enunciado** y la **explicación conceptual**.
2. Intentá resolver en la celda `# TU CÓDIGO AQUÍ`.
3. Si necesitás ayuda, desplegá la **💡 Pista**.
4. Comparate con la **✅ Solución**.

---

In [ ]:
# 🔧 CELDA DE SETUP — Ejecutá esto primero
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (13, 5)
plt.rcParams['axes.spines.top']   = False
plt.rcParams['axes.spines.right'] = False
sns.set_theme(style='whitegrid', palette='muted')

# Statsmodels
import statsmodels.api as sm
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Plotly
try:
    import plotly.graph_objects as go
    import plotly.express as px
    from plotly.subplots import make_subplots
    print('✅ Plotly disponible — los gráficos serán interactivos')
except ImportError:
    print('⚠️  Plotly no instalado. Ejecutá: pip install plotly')

import statsmodels
print(f'✅ Statsmodels {statsmodels.__version__}')
print(f'✅ pandas {pd.__version__} | numpy {np.__version__}')

---
## 📦 Ejercicio 1 — Análisis de series de tiempo con Statsmodels

**Enunciado:** Cargá un dataset de desempeño de ventas a lo largo del tiempo. Realizá un análisis de series temporales para identificar patrones, tendencias y estacionalidad.

### 📖 Explicación conceptual

Una **serie de tiempo** es una secuencia de datos ordenados cronológicamente. En ventas, suelen tener tres componentes:

```
Ventas = Tendencia + Estacionalidad + Ruido

Tendencia:     la dirección general (sube, baja, estable)
Estacionalidad: patrones que se repiten (picos en diciembre, bajas en enero)
Ruido:         variación aleatoria que no se puede predecir
```

**Herramientas de Statsmodels para series de tiempo:**

| Función | Para qué sirve |
|---------|----------------|
| `seasonal_decompose()` | Separa la serie en tendencia, estacionalidad y residuo |
| `adfuller()` | Test de Dickey-Fuller: ¿la serie es estacionaria? |
| `plot_acf()` | Autocorrelación: ¿el valor de hoy depende del de ayer? |
| `ExponentialSmoothing()` | Suavizado exponencial: modelo clásico de forecast |

**¿Qué es una serie estacionaria?**
Una serie es **estacionaria** cuando su media y varianza son constantes en el tiempo (no tiene tendencia ni estacionalidad). Muchos modelos estadísticos requieren estacionariedad.

```
No estacionaria:  /‾‾‾/‾‾‾/  (tiene tendencia ascendente)
Estacionaria:     ∿∿∿∿∿∿∿∿  (oscila alrededor de una media fija)
```

In [ ]:
# 🔧 GENERAMOS EL DATASET DE VENTAS MENSUALES (5 años)
np.random.seed(42)

fechas = pd.date_range('2019-01-01', '2023-12-01', freq='MS')  # 'MS' = inicio de mes
n      = len(fechas)   # 60 meses

# Componente de tendencia (crecimiento lineal + aceleración post-2021)
tendencia = np.linspace(100_000, 180_000, n)
tendencia[24:] *= 1.15   # Boom post-pandemia

# Componente estacional (picos en nov-dic, bajas en ene-feb)
patron_est = np.array([0.85, 0.80, 0.90, 0.95, 1.00, 1.05,
                        1.10, 1.08, 1.02, 1.05, 1.20, 1.40])
estacional = np.tile(patron_est, n // 12 + 1)[:n]

# Ruido aleatorio
ruido = np.random.normal(0, 8_000, n)

# Serie final
ventas = (tendencia * estacional + ruido).round(0)

# Costos (aproximadamente 60% de ventas con algo de variación)
costos   = (ventas * np.random.uniform(0.55, 0.65, n)).round(0)
ganancias = ventas - costos

df_ts = pd.DataFrame({
    'fecha':    fechas,
    'ventas':   ventas,
    'costos':   costos,
    'ganancia': ganancias
})
df_ts.set_index('fecha', inplace=True)

df_ts.to_csv('ventas_temporales.csv')
print(f'Serie de tiempo generada: {len(df_ts)} meses ({df_ts.index[0].date()} → {df_ts.index[-1].date()})')
print(f'\nResumen estadístico:')
print(df_ts.describe().applymap(lambda x: f'${x:,.0f}'))

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# 1. Cargá 'ventas_temporales.csv' con el índice como fecha
# 2. Graficá la serie completa de ventas
# 3. Calculá y graficá la media móvil de 12 meses
# 4. Usá seasonal_decompose() para separar tendencia, estacionalidad y residuo
# 5. Ejecutá el test de Dickey-Fuller para verificar estacionariedad
# 6. Graficá la autocorrelación (ACF)


<details>
<summary>💡 Pista (hacé clic para ver)</summary>

```python
# Cargar con índice de fecha
df = pd.read_csv('ventas_temporales.csv', index_col='fecha', parse_dates=True)

# Media móvil
df['mm_12'] = df['ventas'].rolling(window=12).mean()

# Descomposición
resultado = seasonal_decompose(df['ventas'], model='multiplicative', period=12)
resultado.plot()  # Muestra los 4 componentes

# Test de estacionariedad
resultado_adf = adfuller(df['ventas'].dropna())
print(f'p-valor: {resultado_adf[1]:.4f}')  # p < 0.05 → estacionaria
```
Para ACF: `plot_acf(df['ventas'], lags=24)` muestra las correlaciones con rezagos de hasta 24 meses.

</details>

In [ ]:
# ✅ SOLUCIÓN — Parte A: Exploración de la serie

df = pd.read_csv('ventas_temporales.csv', index_col='fecha', parse_dates=True)
serie = df['ventas']

# Media móvil y bandas
df['mm_12']  = serie.rolling(window=12).mean()
df['mm_3']   = serie.rolling(window=3).mean()
df['std_12'] = serie.rolling(window=12).std()

# Gráfico principal: serie + medias móviles
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(serie.index, serie.values, color='steelblue', alpha=0.5, lw=1.2, label='Ventas mensuales')
ax.plot(df['mm_12'].index, df['mm_12'].values, color='tomato', lw=2.5, label='Media móvil 12 meses')
ax.plot(df['mm_3'].index,  df['mm_3'].values,  color='seagreen', lw=1.5, linestyle='--', label='Media móvil 3 meses')

# Banda de ±1 std
ax.fill_between(
    df['mm_12'].index,
    df['mm_12'] - df['std_12'],
    df['mm_12'] + df['std_12'],
    alpha=0.15, color='tomato', label='±1 desv. estándar'
)

# Anotación del evento post-pandemia
ax.axvline(pd.Timestamp('2021-01-01'), color='orange', lw=1.5, linestyle=':', alpha=0.8)
ax.text(pd.Timestamp('2021-02-01'), serie.max()*0.95, 'Boom\npost-pandemia',
        fontsize=9, color='darkorange', fontweight='bold')

ax.set_title('Serie de tiempo: ventas mensuales 2019–2023', fontsize=13, fontweight='bold')
ax.set_xlabel('Fecha')
ax.set_ylabel('Ventas ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

# Estadísticas anuales
print('📊 Ventas totales por año:')
resumen_anual = df.groupby(df.index.year)['ventas'].agg(['sum','mean','std'])
resumen_anual.columns = ['total', 'promedio_mensual', 'variabilidad']
print(resumen_anual.applymap(lambda x: f'${x:,.0f}'))

In [ ]:
# ✅ SOLUCIÓN — Parte B: Descomposición estacional

print('🔬 Descomposición de la serie temporal (modelo multiplicativo)\n')
print('Modelo multiplicativo: Ventas = Tendencia × Estacionalidad × Residuo')
print('(Apropiado cuando la amplitud de la estacionalidad CRECE con la tendencia)\n')

# Descomposición multiplicativa
descomp = seasonal_decompose(serie, model='multiplicative', period=12)

fig, axes = plt.subplots(4, 1, figsize=(14, 11), sharex=True)

# Serie original
axes[0].plot(serie, color='steelblue', lw=1.5)
axes[0].set_ylabel('Original')
axes[0].set_title('Descomposición de la serie de ventas', fontsize=13, fontweight='bold')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

# Tendencia
axes[1].plot(descomp.trend, color='tomato', lw=2)
axes[1].set_ylabel('Tendencia')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

# Estacionalidad
axes[2].plot(descomp.seasonal, color='seagreen', lw=1.5)
axes[2].axhline(1, color='gray', linestyle='--', alpha=0.5)
axes[2].set_ylabel('Estacionalidad')
axes[2].set_ylim(0.5, 1.6)

# Residuo
axes[3].plot(descomp.resid, color='gray', lw=1, alpha=0.8)
axes[3].axhline(1, color='gray', linestyle='--', alpha=0.5)
axes[3].set_ylabel('Residuo')
axes[3].set_xlabel('Fecha')

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Patrón estacional por mes
patron_df = pd.DataFrame({
    'mes': range(1, 13),
    'factor': descomp.seasonal[:12].values
})
nombres_meses = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']
patron_df['mes_nombre'] = nombres_meses

print('📅 Patrón estacional por mes (>1 = sobre la media, <1 = bajo la media):')
for _, fila in patron_df.iterrows():
    barra = '█' * int((fila['factor'] - 0.7) * 30)
    estado = '🔴' if fila['factor'] < 0.95 else ('🟢' if fila['factor'] > 1.1 else '🟡')
    print(f'  {fila["mes_nombre"]:3s} {estado} {barra} {fila["factor"]:.3f}')

In [ ]:
# ✅ SOLUCIÓN — Parte C: Test de estacionariedad y autocorrelación

print('=' * 60)
print('TEST DE DICKEY-FULLER (ADF) — ¿La serie es estacionaria?')
print('=' * 60)
print('H₀: La serie tiene raíz unitaria (NO es estacionaria)')
print('H₁: La serie ES estacionaria')
print('Criterio: si p-valor < 0.05 → rechazamos H₀ → estacionaria\n')

def test_adf(serie, nombre):
    resultado = adfuller(serie.dropna())
    p_valor   = resultado[1]
    es_est    = p_valor < 0.05
    print(f'Serie: {nombre}')
    print(f'  Estadístico ADF: {resultado[0]:.4f}')
    print(f'  p-valor:         {p_valor:.4f}')
    print(f'  Valores críticos: {{\'1%\': {resultado[4]["1%"]:.2f}, \'5%\': {resultado[4]["5%"]:.2f}}}')
    print(f'  → {"✅ ESTACIONARIA" if es_est else "❌ NO estacionaria (tiene tendencia)"}\n')
    return es_est

# Serie original
test_adf(serie, 'Ventas originales')

# Diferenciación: si no es estacionaria, diferenciamos
serie_diff = serie.diff().dropna()
test_adf(serie_diff, 'Ventas diferenciadas (1er orden)')

# ACF y PACF
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

plot_acf(serie,      lags=24, ax=axes[0,0], title='ACF — Serie original',      color='steelblue')
plot_pacf(serie,     lags=24, ax=axes[0,1], title='PACF — Serie original',     color='steelblue')
plot_acf(serie_diff, lags=24, ax=axes[1,0], title='ACF — Serie diferenciada',  color='seagreen')
plot_pacf(serie_diff,lags=24, ax=axes[1,1], title='PACF — Serie diferenciada', color='seagreen')

for ax in axes.flatten():
    ax.set_xlabel('Rezago (meses)')

plt.suptitle('Funciones de Autocorrelación (ACF) y Autocorrelación Parcial (PACF)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('💡 Cómo interpretar ACF/PACF:')
print('  - Picos en rezago 12, 24... → estacionalidad anual')
print('  - ACF que decae lentamente → serie no estacionaria (tiene tendencia)')
print('  - PACF corta después del rezago p → modelo AR(p) apropiado')

In [ ]:
# ✅ SOLUCIÓN — Parte D: Forecast con Suavizado Exponencial

print('🔮 FORECASTING: Predicción de ventas para los próximos 12 meses\n')

# Dividimos en train (primeros 4 años) y test (último año)
train = serie[:'2022-12-01']
test  = serie['2023-01-01':]
print(f'Train: {len(train)} meses | Test: {len(test)} meses')

# Modelo Holt-Winters (Triple Exponential Smoothing)
# Captura tendencia + estacionalidad multiplicativa
modelo_hw = ExponentialSmoothing(
    train,
    trend='add',               # Tendencia aditiva
    seasonal='multiplicative', # Estacionalidad multiplicativa
    seasonal_periods=12,       # Período estacional = 12 meses
    initialization_method='estimated'
)
ajuste_hw = modelo_hw.fit(optimized=True)  # Optimiza los parámetros automáticamente

# Predicción: 12 meses de test + 12 meses futuros
pred_test   = ajuste_hw.forecast(12)
pred_futuro = ajuste_hw.forecast(24)[12:]  # Próximos 12 meses después del test

# Métricas de error en el test set
mae  = np.mean(np.abs(test.values - pred_test.values))
mape = np.mean(np.abs((test.values - pred_test.values) / test.values)) * 100
rmse = np.sqrt(np.mean((test.values - pred_test.values)**2))

print(f'\n📊 Parámetros del modelo Holt-Winters:')
print(f'  Alpha (nivel):       {ajuste_hw.params["smoothing_level"]:.4f}')
print(f'  Beta (tendencia):    {ajuste_hw.params["smoothing_trend"]:.4f}')
print(f'  Gamma (estacional):  {ajuste_hw.params["smoothing_seasonal"]:.4f}')

print(f'\n📏 Métricas en el set de test (2023):')
print(f'  MAE:  ${mae:,.0f}')
print(f'  RMSE: ${rmse:,.0f}')
print(f'  MAPE: {mape:.2f}% (error porcentual promedio)')

# Gráfico de forecast
fig, ax = plt.subplots(figsize=(15, 6))

# Datos históricos
ax.plot(train.index, train.values, color='steelblue', lw=2, label='Train (2019–2022)')
ax.plot(test.index,  test.values,  color='seagreen',  lw=2, label='Real 2023')

# Predicciones
ax.plot(pred_test.index,   pred_test.values,   color='tomato', lw=2,
        linestyle='--', label=f'Forecast test (MAPE={mape:.1f}%)')
ax.plot(pred_futuro.index, pred_futuro.values, color='darkorange', lw=2,
        linestyle=':', label='Forecast 2024')

# Banda de incertidumbre (±10%)
ax.fill_between(
    pred_futuro.index,
    pred_futuro * 0.90,
    pred_futuro * 1.10,
    alpha=0.2, color='darkorange', label='Intervalo ±10%'
)

# Línea de separación train/test
ax.axvline(pd.Timestamp('2023-01-01'), color='gray', lw=1.5, linestyle=':', alpha=0.7)
ax.text(pd.Timestamp('2023-02-01'), train.max()*1.05, '← Train | Test →',
        fontsize=9, color='gray')

ax.set_title('Forecast de ventas con Holt-Winters (Triple Exponential Smoothing)',
             fontsize=13, fontweight='bold')
ax.set_xlabel('Fecha')
ax.set_ylabel('Ventas ($)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))
ax.legend(loc='upper left')
plt.tight_layout()
plt.show()

# Proyección 2024
print('\n📅 Proyección de ventas para 2024:')
proj_df = pd.DataFrame({'mes': pred_futuro.index.strftime('%b %Y'), 'proyeccion': pred_futuro.values.round(0)})
for _, r in proj_df.iterrows():
    print(f'  {r["mes"]}: ${r["proyeccion"]:,.0f}')
print(f'  TOTAL 2024 estimado: ${proj_df["proyeccion"].sum():,.0f}')

---
## 📦 Ejercicio 2 — Dashboard interactivo de KPIs con Plotly

**Enunciado:** Creá un informe interactivo que muestre métricas clave de rendimiento (KPIs) de una empresa: ingresos, costos y ganancias. Incluí filtros para explorar los datos.

### 📖 Explicación conceptual

**¿Qué son los KPIs?**

Los **KPIs (Key Performance Indicators)** son métricas que miden el desempeño de una empresa en áreas críticas. Un buen KPI debe ser:
- **Específico:** mide una cosa concreta
- **Medible:** tiene un valor numérico
- **Comparable:** puede compararse con un objetivo o período anterior
- **Accionable:** permite tomar decisiones

**KPIs financieros clave:**

| KPI | Fórmula | Qué mide |
|-----|---------|----------|
| Margen bruto | (Ventas - Costo) / Ventas × 100 | Rentabilidad antes de gastos operativos |
| Crecimiento YoY | (Este año - Año anterior) / Año anterior × 100 | Crecimiento respecto al año pasado |
| ROI | (Ganancia / Inversión) × 100 | Retorno sobre la inversión |
| EBITDA | Ganancias + Amortizaciones | Rentabilidad operativa |
| Ticket promedio | Ventas totales / Cantidad de transacciones | Valor por transacción |

**¿Por qué Plotly en vez de Tableau?**
- Plotly es **gratuito** y corre **dentro del notebook**
- Los gráficos son **interactivos** (zoom, hover, filtros)
- Se puede exportar a HTML para compartir
- Los conceptos de diseño de dashboard son **exactamente iguales** a Tableau

In [ ]:
# 🔧 GENERAMOS EL DATASET EMPRESARIAL COMPLETO
np.random.seed(77)

fechas_biz = pd.date_range('2021-01-01', '2023-12-01', freq='MS')
n_biz      = len(fechas_biz)

productos_biz  = ['Software', 'Hardware', 'Servicios', 'Consultoría']
regiones_biz   = ['AMBA', 'Córdoba', 'Rosario', 'Mendoza', 'Resto']
canales        = ['Online', 'Directo', 'Revendedor']

# Generamos ~1000 transacciones
n_trans = 1000
fecha_trans   = np.random.choice(fechas_biz, n_trans)
producto_trans = np.random.choice(productos_biz, n_trans, p=[0.35, 0.25, 0.25, 0.15])

precio_base_biz = {'Software': 5000, 'Hardware': 8000, 'Servicios': 3000, 'Consultoría': 12000}
costo_pct       = {'Software': 0.20, 'Hardware': 0.55, 'Servicios': 0.40, 'Consultoría': 0.30}

ingresos_trans  = np.array([precio_base_biz[p] * np.random.uniform(0.8, 1.3) for p in producto_trans]).round(0)
costos_trans    = np.array([ingresos_trans[i] * costo_pct[producto_trans[i]] * np.random.uniform(0.9, 1.1)
                            for i in range(n_trans)]).round(0)

df_biz = pd.DataFrame({
    'fecha':    fecha_trans,
    'producto': producto_trans,
    'region':   np.random.choice(regiones_biz, n_trans, p=[0.40, 0.20, 0.18, 0.12, 0.10]),
    'canal':    np.random.choice(canales, n_trans, p=[0.45, 0.35, 0.20]),
    'ingresos': ingresos_trans,
    'costos':   costos_trans,
})
df_biz['ganancia']  = df_biz['ingresos'] - df_biz['costos']
df_biz['margen_pct'] = (df_biz['ganancia'] / df_biz['ingresos'] * 100).round(1)
df_biz['año']        = pd.to_datetime(df_biz['fecha']).dt.year
df_biz['mes']        = pd.to_datetime(df_biz['fecha']).dt.month
df_biz['trim']       = pd.to_datetime(df_biz['fecha']).dt.quarter

df_biz.to_csv('kpis_empresa.csv', index=False)
print(f'Dataset empresarial: {len(df_biz):,} transacciones')
print(f'\nResumen financiero total:')
print(f'  Ingresos:  ${df_biz["ingresos"].sum():>12,.0f}')
print(f'  Costos:    ${df_biz["costos"].sum():>12,.0f}')
print(f'  Ganancia:  ${df_biz["ganancia"].sum():>12,.0f}')
print(f'  Margen:    {df_biz["ganancia"].sum()/df_biz["ingresos"].sum()*100:.1f}%')

In [ ]:
# ✏️ TU CÓDIGO AQUÍ
# 1. Calculá los KPIs principales: ingresos totales, costo total, ganancia, margen%
# 2. Calculá el crecimiento año a año (YoY) de ingresos
# 3. Creá un gráfico de barras agrupadas: ingresos, costos y ganancia por año
# 4. Creá un gráfico de líneas de la evolución mensual
# 5. Mostrá la composición de ingresos por producto (torta o barras apiladas)


<details>
<summary>💡 Pista (hacé clic para ver)</summary>

```python
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# KPIs por año
kpi_anual = df_biz.groupby('año').agg(
    ingresos=('ingresos', 'sum'),
    costos=('costos', 'sum'),
    ganancia=('ganancia', 'sum')
)
kpi_anual['margen'] = kpi_anual['ganancia'] / kpi_anual['ingresos'] * 100
kpi_anual['yoy']    = kpi_anual['ingresos'].pct_change() * 100

# Gráfico con Plotly
fig = make_subplots(rows=1, cols=2)
fig.add_trace(go.Bar(name='Ingresos', x=kpi_anual.index, y=kpi_anual['ingresos']), row=1, col=1)
fig.show()
```
En Plotly, `fig.show()` abre el gráfico interactivo. Podés hacer zoom, hover y filtrar con la leyenda.

</details>

In [ ]:
# ✅ SOLUCIÓN — Parte A: Cálculo de KPIs

df_b = pd.read_csv('kpis_empresa.csv', parse_dates=['fecha'])
df_b['año']  = df_b['fecha'].dt.year
df_b['mes']  = df_b['fecha'].dt.month
df_b['trim'] = df_b['fecha'].dt.quarter

# KPIs por año
kpi_anual = df_b.groupby('año').agg(
    ingresos=('ingresos',  'sum'),
    costos=('costos',    'sum'),
    ganancia=('ganancia',  'sum'),
    transacciones=('ingresos', 'count')
).round(0)
kpi_anual['margen_pct']  = (kpi_anual['ganancia'] / kpi_anual['ingresos'] * 100).round(1)
kpi_anual['ticket_prom'] = (kpi_anual['ingresos'] / kpi_anual['transacciones']).round(0)
kpi_anual['yoy_pct']     = kpi_anual['ingresos'].pct_change().mul(100).round(1)

print('=' * 70)
print('📊 KPIs ANUALES DE LA EMPRESA')
print('=' * 70)

for año, row in kpi_anual.iterrows():
    yoy_str = f" ({row['yoy_pct']:+.1f}% vs año anterior)" if not pd.isna(row['yoy_pct']) else ''
    print(f'\n  📅 {año}')
    print(f'    Ingresos:      ${row["ingresos"]:>12,.0f}{yoy_str}')
    print(f'    Costos:        ${row["costos"]:>12,.0f}')
    print(f'    Ganancia:      ${row["ganancia"]:>12,.0f}')
    print(f'    Margen:         {row["margen_pct"]:>10.1f}%')
    print(f'    Ticket prom:   ${row["ticket_prom"]:>12,.0f}')
    print(f'    Transacciones:  {row["transacciones"]:>10,.0f}')

# Variación entre 2021 y 2023
crecim_total = (kpi_anual.loc[2023,'ingresos'] / kpi_anual.loc[2021,'ingresos'] - 1) * 100
print(f'\n🚀 Crecimiento total 2021→2023: {crecim_total:+.1f}%')

In [ ]:
# ✅ SOLUCIÓN — Parte B: Dashboard con Plotly (gráficos interactivos)

import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

colores_dash = {'ingresos': '#2196F3', 'costos': '#F44336', 'ganancia': '#4CAF50'}
años         = kpi_anual.index.tolist()

# Dashboard principal: 2x3
fig = make_subplots(
    rows=2, cols=3,
    subplot_titles=[
        'Ingresos, Costos y Ganancia por año',
        'Evolución mensual de ingresos',
        'Participación por producto (%)',
        'Margen bruto por año (%)',
        'Ingresos por región y producto',
        'Crecimiento YoY por trimestre'
    ],
    specs=[
        [{'type': 'bar'},     {'type': 'scatter'}, {'type': 'pie'}],
        [{'type': 'scatter'}, {'type': 'bar'},     {'type': 'bar'}]
    ]
)

# --- Panel 1: Barras agrupadas ingresos/costos/ganancia por año ---
for metrica, color in colores_dash.items():
    fig.add_trace(go.Bar(
        name=metrica.capitalize(), x=años,
        y=kpi_anual[metrica].values,
        marker_color=color, opacity=0.85,
        text=[f'${v/1e6:.1f}M' for v in kpi_anual[metrica].values],
        textposition='outside'
    ), row=1, col=1)

# --- Panel 2: Evolución mensual ---
mensual = df_b.groupby('fecha')[['ingresos','costos','ganancia']].sum().reset_index()
for metrica, color in colores_dash.items():
    fig.add_trace(go.Scatter(
        x=mensual['fecha'], y=mensual[metrica],
        name=metrica, line=dict(color=color, width=2),
        showlegend=False
    ), row=1, col=2)

# --- Panel 3: Torta por producto ---
por_prod = df_b.groupby('producto')['ingresos'].sum()
fig.add_trace(go.Pie(
    labels=por_prod.index, values=por_prod.values,
    hole=0.35,
    marker_colors=px.colors.qualitative.Set2,
    showlegend=True
), row=1, col=3)

# --- Panel 4: Margen % por año ---
fig.add_trace(go.Scatter(
    x=años, y=kpi_anual['margen_pct'].values,
    mode='lines+markers+text',
    text=[f'{v:.1f}%' for v in kpi_anual['margen_pct'].values],
    textposition='top center',
    line=dict(color='#4CAF50', width=3),
    marker=dict(size=12), showlegend=False
), row=2, col=1)

# --- Panel 5: Ingresos por región (barras apiladas) ---
por_reg_prod = df_b.groupby(['region','producto'])['ingresos'].sum().reset_index()
colores_prod = px.colors.qualitative.Set2
for i, prod in enumerate(productos_biz):
    sub = por_reg_prod[por_reg_prod['producto'] == prod]
    fig.add_trace(go.Bar(
        name=prod, x=sub['region'], y=sub['ingresos'],
        marker_color=colores_prod[i], showlegend=False
    ), row=2, col=2)

# --- Panel 6: Crecimiento YoY por trimestre ---
trim_anual = df_b.groupby(['año','trim'])['ingresos'].sum().reset_index()
trim_anual['label'] = trim_anual['año'].astype(str) + '-Q' + trim_anual['trim'].astype(str)
trim_anual['yoy']   = trim_anual.groupby('trim')['ingresos'].pct_change() * 100
trim_anual_filt     = trim_anual.dropna(subset=['yoy'])

fig.add_trace(go.Bar(
    x=trim_anual_filt['label'],
    y=trim_anual_filt['yoy'],
    marker_color=['#4CAF50' if v > 0 else '#F44336' for v in trim_anual_filt['yoy']],
    text=[f'{v:.1f}%' for v in trim_anual_filt['yoy']],
    textposition='outside',
    showlegend=False
), row=2, col=3)

# Layout general
fig.update_layout(
    title_text='📊 Dashboard de KPIs Empresariales — 2021 a 2023',
    title_font_size=18,
    height=750,
    barmode='group',
    template='plotly_white',
    legend=dict(orientation='h', y=1.02, x=0),
    font=dict(size=11)
)

# Formato de ejes en dólares
for col in [1, 2]:
    fig.update_yaxes(tickprefix='$', tickformat=',.0f', row=1, col=col)
fig.update_yaxes(tickprefix='$', tickformat=',.0f', row=2, col=2)
fig.update_yaxes(ticksuffix='%', row=2, col=1)
fig.update_yaxes(ticksuffix='%', row=2, col=3)

fig.show()
print('✅ Dashboard generado. Podés interactuar: zoom, hover, clic en la leyenda para filtrar.')

In [ ]:
# ✅ SOLUCIÓN — Parte C: Análisis por canal y segmento (drill-down)

print('🔍 Análisis de rentabilidad por segmento\n')

# Tabla de rentabilidad por producto y canal
rentab = df_b.groupby(['producto', 'canal']).agg(
    ingresos=('ingresos',   'sum'),
    ganancia=('ganancia',   'sum'),
    transacciones=('ingresos', 'count')
).round(0)
rentab['margen_%']     = (rentab['ganancia'] / rentab['ingresos'] * 100).round(1)
rentab['ticket_prom']  = (rentab['ingresos'] / rentab['transacciones']).round(0)
rentab = rentab.sort_values('margen_%', ascending=False)

print('Rentabilidad por producto y canal:')
print(rentab[['ingresos','ganancia','margen_%','ticket_prom']]
      .to_string(formatter={
          'ingresos':    lambda x: f'${x:,.0f}',
          'ganancia':    lambda x: f'${x:,.0f}',
          'margen_%':    lambda x: f'{x:.1f}%',
          'ticket_prom': lambda x: f'${x:,.0f}'
      }))

# Gráfico de burbujas: producto vs canal vs margen
fig_bub = px.scatter(
    rentab.reset_index(),
    x='ingresos', y='margen_%',
    size='transacciones', color='producto',
    symbol='canal', hover_name='producto',
    hover_data={'canal': True, 'ingresos': ':$,.0f', 'ganancia': ':$,.0f'},
    title='Análisis de segmentos: Ingresos vs Margen (tamaño = volumen de transacciones)',
    labels={'ingresos': 'Ingresos totales ($)', 'margen_%': 'Margen bruto (%)', 'transacciones': 'Transacciones'},
    template='plotly_white',
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig_bub.add_hline(y=df_b['margen_pct'].mean(), line_dash='dash', line_color='gray',
                   annotation_text=f'Margen promedio: {df_b["margen_pct"].mean():.1f}%')
fig_bub.update_layout(height=500)
fig_bub.show()

print('\n💡 Insights del análisis:')
mejor_margen = rentab['margen_%'].idxmax()
peor_margen  = rentab['margen_%'].idxmin()
print(f'  🟢 Segmento más rentable: {mejor_margen} → {rentab.loc[mejor_margen,"margen_%"]:.1f}%')
print(f'  🔴 Segmento menos rentable: {peor_margen} → {rentab.loc[peor_margen,"margen_%"]:.1f}%')

---
## 🏆 Resumen del notebook

| Ejercicio | Tema | Herramientas |
|-----------|------|--------------|
| 1A | Serie de tiempo + medias móviles | `pandas`, `rolling()`, `matplotlib` |
| 1B | Descomposición estacional | `seasonal_decompose`, modelo multiplicativo |
| 1C | Estacionariedad + ACF/PACF | `adfuller`, `plot_acf`, `plot_pacf` |
| 1D | Forecast con Holt-Winters | `ExponentialSmoothing`, MAE, MAPE |
| 2A | Cálculo de KPIs financieros | `groupby`, crecimiento YoY, margen |
| 2B | Dashboard interactivo | `plotly`, `make_subplots`, torta, barras, líneas |
| 2C | Análisis de segmentos | Gráfico de burbujas, drill-down por producto/canal |

### 🔑 Conceptos clave de Business Analytics

| Concepto | Descripción |
|----------|-------------|
| **Tendencia** | Dirección general de la serie en el largo plazo |
| **Estacionalidad** | Patrón repetitivo en períodos fijos (mensual, trimestral, anual) |
| **Estacionariedad** | La serie no tiene tendencia ni estacionalidad (media y varianza constantes) |
| **MAPE** | Error porcentual promedio: el más intuitivo para comunicar forecast a negocio |
| **Margen bruto** | % de cada venta que queda como ganancia antes de gastos operativos |
| **YoY** | Year-on-Year: comparación con el mismo período del año anterior |

### 💪 Desafíos extra
1. **Ej. 1:** Aplicá un modelo SARIMA en vez de Holt-Winters y comparalos por MAPE.
2. **Ej. 1:** Simulá un shock externo (caída del 30% en el mes 37) y analizá cómo afecta el forecast.
3. **Ej. 2:** Agregá un gráfico de waterfall para mostrar cómo se pasa de ingresos a ganancia neta.
4. **Ej. 2:** Calculá el **churn mensual** (% de clientes que no repiten compra) y graficalo.
5. **Ambos:** Exportá el dashboard como HTML con `fig.write_html('dashboard.html')` y abrilo en el browser.